# ESI Baseline Comparison

This notebook compares classical models and two lightweight pretrained
encoders on a balanced 4,000-record training split and a patient-separated
500-record test split. Matching MedGemma results are included only after that
profile has been run.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "prepare_training_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline_utils import (
    ESI_LEVELS,
    NUMERIC_FEATURES,
    TEXT_FEATURE,
    compute_metrics,
    fit_transformer,
    load_split,
    serialize_record,
    summary_table,
    test_fingerprint,
)

SPLIT_DIR = PROJECT_ROOT / "data" / "finetune_mimic_balanced_large"
MEDGEMMA_DIR = PROJECT_ROOT / "outputs" / "notebook_mimic_balanced_large_label_scoring"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TRANSFORMERS = True
SEED = 42

## 1. Load and verify the comparison data

All baseline models use 4,000 balanced training encounters and the same 500
balanced test encounters. Patients cannot cross splits. Matching MedGemma
results are loaded only when the optional `balanced_large` profile exists.

In [ ]:
train = load_split(SPLIT_DIR / "train.jsonl")
test = load_split(SPLIT_DIR / "test.jsonl")
fingerprint = test_fingerprint(test)

assert set(train["subject_id"]).isdisjoint(test["subject_id"])

display(pd.DataFrame({
    "train": train["label"].value_counts().sort_index(),
    "test": test["label"].value_counts().sort_index(),
}).rename_axis("ESI level"))
print(f"Verified {len(train)} training and {len(test)} test records.")

expected_records = [
    (int(row.subject_id), int(row.stay_id), int(row.label))
    for row in test.itertuples(index=False)
]
medgemma_runs = []
for name, filename in [
    ("MedGemma base", "base_evaluation.json"),
    ("MedGemma QLoRA adapter", "adapter_evaluation.json"),
]:
    path = MEDGEMMA_DIR / filename
    if not path.exists():
        continue
    run = json.loads(path.read_text())
    assert run["test_fingerprint"] == fingerprint
    saved_records = [
        (int(row["subject_id"]), int(row["stay_id"]), int(row["actual"]))
        for row in run["predictions"]
    ]
    assert saved_records == expected_records
    medgemma_runs.append((name, run))

if medgemma_runs:
    print(f"Loaded {len(medgemma_runs)} matching MedGemma result set(s).")
else:
    print("No matching large-split MedGemma results yet; baseline-only run.")

## 2. Classical baselines

- **Always ESI 3** gives a trivial reference point.
- **Histogram gradient boosting** uses only vitals and pain.
- **Logistic regression** and **linear SVM** use TF-IDF chief-complaint
  features together with the numeric fields.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def numeric_features():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUMERIC_FEATURES),
    ])


def mixed_features():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), TEXT_FEATURE),
    ])


models = [
    ("Always ESI 3", DummyClassifier(strategy="constant", constant=3)),
    ("Gradient boosting", Pipeline([
        ("features", numeric_features()),
        ("model", HistGradientBoostingClassifier(random_state=SEED)),
    ])),
    ("TF-IDF + logistic regression", Pipeline([
        ("features", mixed_features()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])),
    ("TF-IDF + linear SVM", Pipeline([
        ("features", mixed_features()),
        ("model", LinearSVC(random_state=SEED)),
    ])),
]

results = []
for name, model in models:
    started = time.perf_counter()
    model.fit(train, train["label"])
    fit_seconds = time.perf_counter() - started

    started = time.perf_counter()
    predictions = model.predict(test).astype(int).tolist()
    predict_seconds = time.perf_counter() - started

    results.append({
        "model": name,
        "predictions": predictions,
        "metrics": compute_metrics(test["label"], predictions),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
    })

display(summary_table(results).round(3))

## 3. Lightweight pretrained transformers

DistilBERT and BioClinicalBERT receive the same serialized triage fields.
Each model is fine-tuned once for three epochs with seed 42. Set
`RUN_TRANSFORMERS = False` to skip this section.

In [ ]:
if RUN_TRANSFORMERS:
    train_texts = [serialize_record(row) for _, row in train.iterrows()]
    test_texts = [serialize_record(row) for _, row in test.iterrows()]

    transformer_models = [
        ("DistilBERT", "distilbert-base-uncased"),
        ("BioClinicalBERT", "emilyalsentzer/Bio_ClinicalBERT"),
    ]
    for display_name, model_id in transformer_models:
        run = fit_transformer(
            model_id,
            train_texts,
            train["label"],
            test_texts,
            seed=SEED,
        )
        predictions = run["predictions"]
        results.append({
            "model": display_name,
            "predictions": predictions,
            "metrics": compute_metrics(test["label"], predictions),
            "fit_seconds": run["fit_seconds"],
            "predict_seconds": run["predict_seconds"],
            "device": run["device"],
        })
        print(f"Finished {display_name} on {run['device']}.")
else:
    print("Transformer baselines skipped.")

## 4. Add matching MedGemma results when available

This section remains empty until the MedGemma `balanced_large` profile has
been evaluated on the identical test fingerprint.

In [ ]:
for name, run in medgemma_runs:
    predictions = [row["prediction"] for row in run["predictions"]]
    results.append({
        "model": name,
        "predictions": predictions,
        "metrics": compute_metrics(test["label"], predictions),
        "fit_seconds": None,
        "predict_seconds": None,
    })

## 5. Compare results

ESI 1 is most urgent. A numerically larger prediction is under-triage; a
numerically smaller prediction is over-triage. Severe under-triage means an
actual ESI 1 or 2 was predicted as ESI 4 or 5.

In [ ]:
summary = summary_table(results)
display(summary.round(3))

recall = pd.DataFrame({
    result["model"]: result["metrics"]["recall_by_esi"]
    for result in results
}).T
recall.columns = [f"ESI {level} recall" for level in ESI_LEVELS]
display(recall.round(3))

In [ ]:
for result in results:
    matrix = pd.DataFrame(
        result["metrics"]["confusion_matrix_labels_1_to_5"],
        index=[f"actual {level}" for level in ESI_LEVELS],
        columns=[f"predicted {level}" for level in ESI_LEVELS],
    )
    print(result["model"])
    display(matrix)

## 6. Save results

The output contains aggregate metrics and model predictions only. The 100-case
test is useful for a smoke comparison, but it is too small for clinical claims.

In [ ]:
output = {
    "test_fingerprint": fingerprint,
    "train_examples": len(train),
    "test_examples": len(test),
    "seed": SEED,
    "results": results,
}
output_path = OUTPUT_DIR / "baseline_results_large_balanced.json"
output_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")